# 0.0 — Resumen Ejecutivo del Proyecto
**Trabajo Final de Aprendizaje Automático — Segmentación de Clientes**

**Autor:** Ian Gallego | **Politécnico Malvinas Argentinas** | 2026

---

## 🎯 Pregunta de negocio

> *¿Cómo se pueden agrupar los clientes de un e-commerce de ferretería para diseñar
> estrategias de marketing personalizadas, en lugar de seguir enviando promociones masivas
> uniformes a toda la base?*

## 🏁 Resultado principal

Se identificaron **4 segmentos** mediante K-Means sobre features RFM deflactadas:

| Segmento | Clientes | % Base | % Facturación (real) | Estrategia |
|----------|----------|--------|----------------------|------------|
| **🏆 Campeones (B2B)** | 154 | 15.4 % | **76.7 %** | Programa premium, cuenta corriente, gestor dedicado |
| 🛒 Activos Recientes | 306 | 30.6 % | 16.4 % | Up-sell, programa de puntos |
| ⚠️ Esporádicos / En Riesgo | 322 | 32.2 % | 5.6 % | Campaña de re-activación |
| 💤 Perdidos | 217 | 21.7 % | 1.2 % | Win-back de bajo costo |

**Lectura clave:** un Pareto comercial muy marcado — el **15 % de los clientes
(Campeones B2B) explica el 77 % de la facturación real**. Esto justifica un trato
diferenciado de marketing.

## 🗂️ Mapa de notebooks

El proyecto sigue la estructura **Cookiecutter Data Science (CCDS)**. Los notebooks se ejecutan en orden:

| # | Notebook | Qué hace |
|---|----------|----------|
| **1** | `1.0-ig-eda-y-limpieza.ipynb` | EDA inicial, normaliza variantes en `localidad`/`medio_pago`, dropea nulos críticos, mantiene outliers B2B legítimos 
| **2** | `2.0-ig-features-rfm.ipynb` | Agrupa por `id_cliente` -> tabla RFM. **Aplica deflactación por inflación** (clave para que clientes recientes vs antiguos sean comparables) 
| **3** | `3.0-ig-eda-rfm.ipynb` | EDA de la tabla cliente: correlaciones (justifica RFM puro), outliers, B2B vs B2C heurístico 
| **4** | `4.0-ig-clustering.ipynb` ⭐ | **Notebook principal.** Aplica K-Means, DBSCAN, Jerárquico, GMM. Compara con Silhouette/DBI/CH. Decide K=4 con justificación 
| **5** | `5.0-ig-interpretacion-y-estrategias.ipynb` | Caracteriza segmentos en unidades originales, nombramiento comercial, estrategia por segmento, Pareto 



## 🧠 Decisiones metodológicas clave

| Decisión | Justificación |
|----------|---------------|
| **Features de clustering**: solo `recency`, `frequency`, `monetary_log` | `ticket_promedio` = M/F -> multicolineal. Las complementarias quedan fuera del clustering pero se usan para caracterizar |
| **K = 4** (no K=2 que maximizaría Silhouette) | K=2 produce solo "activos vs inactivos" — insuficiente para personalización. K=4 es el estándar RFM |
| **K-Means como modelo de producción** (validado por Jerárquico y GMM) | Simple, interpretable y con `.predict()` nativo. GMM también soporta predict pero K-Means es el estándar RFM |
| **Deflactación de `monto_total`** | El dataset modela inflación a 24 meses (50%/anual). Sin deflactar, clientes recientes parecerían artificialmente más valiosos |
| **Caracterización en unidades originales** | Los "$3.7M ARS" se entienden; los z-scores no |

## 🚀 Cómo reproducir el pipeline completo

```bash
# Opción 1: ejecutar todos los notebooks en orden
cd TrabajoFinalApAut
for nb in notebooks/[1-5].*.ipynb; do
    jupyter nbconvert --to notebook --execute "$nb" --inplace
done

# Opción 2 (recomendada): usar el Makefile
make all

# Generar el PDF del notebook principal
make report
```

## 📁 Outputs que se generan

| Directorio / archivo | Contenido |
|----------------------|-----------|
| `data/interim/transacciones_limpias.parquet` | Dataset limpio (10.132 transacciones) |
| `data/processed/clientes_rfm.parquet` | Tabla cliente con RFM + complementarias (999 clientes) |
| `data/processed/clientes_segmentados_nombrados.parquet` | + etiqueta de cluster + segmento comercial |
| `models/modelo_final_kmeans.joblib` | Modelo K-Means de producción |
| `models/scaler.joblib` | StandardScaler ajustado |
| `models/deflactor.joblib` | Serie mensual de deflactor (para clientes nuevos) |
| `reports/4.0-ig-clustering.pdf` | PDF del notebook principal |
| `reports/figures/*.png` | 18+ figuras (codo, dendrograma, Pareto, perfiles, etc.) |

## 📚 Referencias

- `references/diccionario_datos.md` — diccionario de datos del CSV crudo
- `references/glosario.md` — glosario de términos técnicos (RFM, Silhouette, etc.)
- `README.md` — descripción extensa del proyecto y metodología